# 📤 Export Data for Frontend Dashboard

This notebook exports processed results to JSON format for the web dashboard.

**Exports:**
- Area statistics by year
- Trend data (2020-2025)
- Ecological summaries
- Classification maps

In [1]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Setup paths
frontend_data = Path('../frontend/data')
frontend_data.mkdir(exist_ok=True)

print("✅ Export environment ready")

✅ Export environment ready


## 1. Calculate Area Statistics

Calculate km² for each class based on pixel counts and resolution.

In [2]:
# Mock data for demonstration - replace with actual model predictions
# Pixel resolution: 10m x 10m = 100 m² per pixel
# 1 km² = 1,000,000 m² = 10,000 pixels

PIXEL_TO_KM2 = 100 / 1_000_000  # 100 m² per pixel to km²

# Sample pixel counts by year (replace with actual predictions)
year_data = {
    2020: {'Seagrass': 4201000, 'Sand': 1220000, 'Water': 8700000, 'Landmass': 780000},
    2021: {'Seagrass': 4300000, 'Sand': 1210000, 'Water': 8750000, 'Landmass': 781000},
    2022: {'Seagrass': 4400000, 'Sand': 1208000, 'Water': 8800000, 'Landmass': 782000},
    2023: {'Seagrass': 4502000, 'Sand': 1205000, 'Water': 8903000, 'Landmass': 781000},
    2024: {'Seagrass': 4550000, 'Sand': 1200000, 'Water': 8920000, 'Landmass': 781000},
    2025: {'Seagrass': 4600000, 'Sand': 1195000, 'Water': 8950000, 'Landmass': 781000}
}

def calculate_areas(pixel_counts):
    """Convert pixel counts to km²"""
    return {cls: round(count * PIXEL_TO_KM2, 1) for cls, count in pixel_counts.items()}

def calculate_change(current, previous):
    """Calculate percentage change"""
    if previous == 0:
        return 0.0
    return round(((current - previous) / previous) * 100, 1)

# Process each year
area_stats = {}
years = sorted(year_data.keys())

for i, year in enumerate(years):
    areas = calculate_areas(year_data[year])
    stats = []
    
    for category, area in areas.items():
        # Calculate change from previous year
        if i > 0:
            prev_year = years[i-1]
            prev_area = calculate_areas(year_data[prev_year])[category]
            change = calculate_change(area, prev_area)
        else:
            change = 0.0
        
        stats.append({
            'category': category,
            'area': area,
            'change': change
        })
    
    area_stats[str(year)] = stats

print("✅ Area statistics calculated")
print(f"Years: {years}")

✅ Area statistics calculated
Years: [2020, 2021, 2022, 2023, 2024, 2025]


## 2. Generate Trend Data

Prepare time-series data for charts.

In [3]:
# Extract trend data for line chart
trend_data = {
    'years': years,
    'seagrass': [],
    'sand': [],
    'water': [],
    'landmass': []
}

for year in years:
    areas = calculate_areas(year_data[year])
    trend_data['seagrass'].append(areas['Seagrass'])
    trend_data['sand'].append(areas['Sand'])
    trend_data['water'].append(areas['Water'])
    trend_data['landmass'].append(areas['Landmass'])

print("✅ Trend data prepared")

✅ Trend data prepared


## 3. Create Ecological Summaries

Generate text summaries for each year.

In [4]:
# Generate summaries based on trends
eco_summaries = {}

for i, year in enumerate(years):
    if i == 0:
        eco_summaries[str(year)] = f"Baseline year established for coastal monitoring. Initial seagrass coverage: {trend_data['seagrass'][i]} km²."
    else:
        sg_change = trend_data['seagrass'][i] - trend_data['seagrass'][i-1]
        sand_change = trend_data['sand'][i] - trend_data['sand'][i-1]
        
        sg_status = "increased" if sg_change > 0 else "decreased" if sg_change < 0 else "remained stable"
        sand_status = "expanded" if sand_change > 0 else "contracted" if sand_change < 0 else "remained stable"
        
        eco_summaries[str(year)] = f"""During {year}, seagrass coverage <b>{sg_status}</b> by {abs(sg_change):.1f} km² compared to {year-1}. 
        Sand shoals {sand_status} by {abs(sand_change):.1f} km². 
        Overall, the coastal ecosystem shows {'positive' if sg_change > 0 else 'concerning'} trends in biodiversity indicators."""

print("✅ Ecological summaries generated")

✅ Ecological summaries generated


## 4. Export to JSON Files

Save all data as JSON for frontend consumption.

In [5]:
# Export area statistics
with open(frontend_data / 'area_stats.json', 'w') as f:
    json.dump(area_stats, f, indent=2)
print("✅ Exported: area_stats.json")

# Export trend data
with open(frontend_data / 'trend_data.json', 'w') as f:
    json.dump(trend_data, f, indent=2)
print("✅ Exported: trend_data.json")

# Export ecological summaries
with open(frontend_data / 'eco_summaries.json', 'w') as f:
    json.dump(eco_summaries, f, indent=2)
print("✅ Exported: eco_summaries.json")

print("\n🎉 All data exported to frontend/data/")

✅ Exported: area_stats.json
✅ Exported: trend_data.json
✅ Exported: eco_summaries.json

🎉 All data exported to frontend/data/


## 5. Optional: Export Classification Maps

Copy or convert classification maps for web display.

In [6]:
# If you have classification maps as images, copy them
# Example:
# import shutil
# for year in years:
#     src = f'../results/classification_map_{year}.png'
#     dst = f'../frontend/data/map_{year}.png'
#     if os.path.exists(src):
#         shutil.copy(src, dst)
#         print(f"✅ Copied map for {year}")

print("📝 Note: Update this cell to copy actual classification maps if available")

📝 Note: Update this cell to copy actual classification maps if available


## ✅ Export Complete

Frontend can now load data from:
- `frontend/data/area_stats.json`
- `frontend/data/trend_data.json`
- `frontend/data/eco_summaries.json`

**Next Steps:**
1. Update `app.js` to fetch from JSON files
2. Replace mock data with real predictions from notebook 04
3. Export classification maps if available